# V12.3: Spectral-Native Architecture

## The Problem with V12.0-V12.2

Every previous version jumped between spectral and spatial domains inside each block:
```
spectral -> irfft -> SSM(spatial) -> transport -> irfft -> settle(spatial) -> MLP(spatial) -> rfft -> sparsify
```
Three FFT round-trips per block. Spatial was the computation space, spectral was a waypoint.
The model learned to ignore spectral because all real work happened in spatial domain.

## V12.3: Live in Spectral

Tokens ARE spectral patterns — constellations of active modes in Fourier space.
The SSM and MLP learn to **adjust spectral coordinates directly**.
Spatial reconstruction (irfft) happens **only at the decoder**.

```
spectral -> SSM(spectral) -> SpectralInteraction -> Transport -> MLP(spectral) -> residual + sparsify
```

Zero irfft inside blocks. The model thinks, remembers, and computes in spectral domain.

| Component | V12.2 | V12.3 |
|-----------|-------|-------|
| SSM input | spatial (irfft'd) | spectral coordinates (real+imag) |
| MLP input | spatial (irfft'd) | spectral coordinates (real+imag) |
| Hopfield settling | spatial domain, 2 Langevin steps | **removed** — MLP handles nonlinear transforms |
| irfft per block | 2-3 round-trips | **zero** — only at decoder |
| Residual stream | spatial | spectral |
| Sequence length | 128 | 512 |

## Literature

- **Donoho-Stark 1989**: Spectral sparsity guarantees spatial coverage. 6 modes per 32-dim subbundle.
- **Candes 2006**: Compressed sensing at the information-theoretic limit.
- **Shi 2025**: Reaction-diffusion — spectral interaction IS the reaction, transport IS the diffusion.
- **Li 2026 (DMHN)**: Context reshapes the attractor manifold — our SSM context reshapes spectral interactions.

## Preserved Non-Negotiables

1. **Sparse in spectral** — 6/17 modes per subbundle, enforced every block
2. **Field reconstruction IS the irfft** — at decoder output only
3. **Context warps the spectral metric** — D, A are context-dependent
4. **No pairwise token attention**

## Removed

- Hopfield memory and Langevin settling (spatial-domain concepts)
- All intra-block irfft/rfft calls
- Spatial residual stream

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
import math
import time
import os

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Device: {device}")

DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
DATA_PATH = os.path.join(os.getcwd(), "tiny_shakespeare.txt")
if not os.path.exists(DATA_PATH):
    import urllib.request
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

with open(DATA_PATH) as f:
    text = f.read()
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
split = int(0.9 * len(data))
train_data, val_data = data[:split], data[split:]
print(f"Tiny Shakespeare: {len(data):,} chars, vocab {vocab_size}")

Device: mps
Tiny Shakespeare: 1,115,394 chars, vocab 65


In [2]:
@dataclass
class V12_3Config:
    fiber_dim: int = 256
    n_subbundles: int = 8
    spectral_sparsity: int = 6       # 6/17 = 35% active
    vocab_size: int = 65
    max_seq_len: int = 512
    n_blocks: int = 6
    context_dim: int = 128
    mlp_ratio: float = 1.5
    learning_rate: float = 3e-3
    min_lr: float = 3e-4
    warmup_steps: int = 750
    lr_hold_steps: int = 3250
    dropout: float = 0.1
    batch_size: int = 16
    seq_len: int = 512
    max_steps: int = 10000
    eval_interval: int = 500
    eval_steps: int = 10

    @property
    def subbundle_dim(self):
        return self.fiber_dim // self.n_subbundles   # 32

    @property
    def spectral_half_dim(self):
        return self.subbundle_dim // 2 + 1           # 17

    @property
    def spectral_dim(self):
        return self.n_subbundles * self.spectral_half_dim  # 136

    @property
    def spectral_real_dim(self):
        return self.spectral_dim * 2                 # 272 (real+imag concat)

    @property
    def total_active_modes(self):
        return self.n_subbundles * self.spectral_sparsity  # 48

    @property
    def mlp_hidden(self):
        return int(self.spectral_real_dim * self.mlp_ratio)  # 408


config = V12_3Config(vocab_size=vocab_size)
print(f"Spectral dim: {config.spectral_dim} complex = {config.spectral_real_dim} real")
print(f"Sparsity: {config.spectral_sparsity}/{config.spectral_half_dim} per subbundle "
      f"= {100*config.spectral_sparsity/config.spectral_half_dim:.0f}% active")
print(f"Blocks: {config.n_blocks}, Context: {config.context_dim}, MLP hidden: {config.mlp_hidden}")
print(f"Seq len: {config.seq_len}, Batch: {config.batch_size}")

def get_batch(split_data, cfg):
    ix = torch.randint(0, len(split_data) - cfg.seq_len - 1, (cfg.batch_size,))
    return torch.stack([split_data[i:i+cfg.seq_len] for i in ix]).to(device)

Spectral dim: 136 complex = 272 real
Sparsity: 6/17 per subbundle = 35% active
Blocks: 6, Context: 128, MLP hidden: 408
Seq len: 512, Batch: 16


## Architecture: Everything Lives in Spectral Domain

The spectral state is the **only** state that persists across blocks.
Complex 136-dim (272 real) with 6/17 modes active per subbundle.
SSM, MLP, interaction, transport — all operate on spectral coordinates.
irfft happens once, at the decoder.

In [3]:
def spectral_sparsify(x_complex, cfg):
    """Per-subbundle top-k sparsification in spectral domain."""
    shape = x_complex.shape
    x_subs = x_complex.reshape(*shape[:-1], cfg.n_subbundles, cfg.spectral_half_dim)
    with torch.no_grad():
        mags = x_subs.abs()
        gt_count = (mags.unsqueeze(-1) < mags.unsqueeze(-2)).sum(dim=-1)
        mask = (gt_count < cfg.spectral_sparsity).float()
    return (x_subs * mask).reshape(shape)


def spectral_to_spatial(x_spectral, cfg):
    """irfft: spectral -> spatial. Used ONLY at decoder."""
    shape = x_spectral.shape[:-1]
    subs = x_spectral.reshape(*shape, cfg.n_subbundles, cfg.spectral_half_dim)
    return torch.fft.irfft(subs, n=cfg.subbundle_dim, dim=-1).reshape(*shape, cfg.fiber_dim)


def to_real(x_complex):
    """Complex -> real: cat(real, imag) along last dim."""
    return torch.cat([x_complex.real, x_complex.imag], dim=-1)


def to_complex(x_real, spectral_dim):
    """Real -> complex: split last dim into real + imag."""
    return torch.complex(x_real[..., :spectral_dim], x_real[..., spectral_dim:])


def parallel_associative_scan(A, B):
    _, T, _ = A.shape
    a, b = A, B
    for d in range(int(math.ceil(math.log2(T)))):
        step = 2 ** d
        if step >= T:
            break
        b = torch.cat([b[:, :step, :],
                        a[:, step:, :] * b[:, :-step, :] + b[:, step:, :]], dim=1)
        a = torch.cat([a[:, :step, :],
                        a[:, step:, :] * a[:, :-step, :]], dim=1)
    return b


# Verify
_t = torch.randn(2, 4, config.spectral_real_dim)
_c = to_complex(_t, config.spectral_dim)
_b = to_real(_c)
print(f"real<->complex round-trip error: {(_t - _b).abs().max():.2e}")
print(f"spectral_real_dim={config.spectral_real_dim} (136 complex = 272 real)")

real<->complex round-trip error: 0.00e+00
spectral_real_dim=272 (136 complex = 272 real)


In [4]:
class SpectralTokenEmbedding(nn.Module):
    """Tokens -> sparse spectral patterns (constellations of active modes)."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.mag_embedding = nn.Embedding(cfg.vocab_size, cfg.spectral_dim)
        self.phase_embedding = nn.Embedding(cfg.vocab_size, cfg.spectral_dim)
        nn.init.uniform_(self.phase_embedding.weight, -math.pi, math.pi)
        freqs = torch.zeros(cfg.spectral_dim)
        for k in range(cfg.n_subbundles):
            off = k * cfg.spectral_half_dim
            freqs[off:off+cfg.spectral_half_dim] = (
                2 * math.pi * torch.fft.rfftfreq(cfg.subbundle_dim, d=1.0))
        self.register_buffer('freqs', freqs)

    def forward(self, token_ids):
        B, T = token_ids.shape
        mag = self.mag_embedding(token_ids)
        phase = self.phase_embedding(token_ids)
        pos = torch.arange(T, device=token_ids.device).float()
        total_phase = phase + (pos.unsqueeze(-1) * self.freqs).unsqueeze(0)
        spectral = mag * torch.exp(1j * total_phase)
        return spectral_sparsify(spectral, self.cfg)

In [5]:
class SpectralNorm(nn.Module):
    """LayerNorm on real+imag concatenation. Preserves complex structure."""

    def __init__(self, spectral_dim):
        super().__init__()
        self.norm = nn.LayerNorm(spectral_dim * 2)
        self.spectral_dim = spectral_dim

    def forward(self, x):
        return to_complex(self.norm(to_real(x)), self.spectral_dim)

In [6]:
class SpectralSSM(nn.Module):
    """SSM that reads spectral coordinates directly.
    Input: complex spectral state -> concat real+imag -> 272-dim real.
    Output: real-valued context (128-dim) for gating transport + interaction.

    The SSM learns sequential patterns FROM spectral coordinates —
    which modes are active, how they evolve over the sequence."""

    def __init__(self, cfg):
        super().__init__()
        rd = cfg.spectral_real_dim  # 272
        cd = cfg.context_dim        # 128
        self.A_proj = nn.Linear(rd, cd)
        self.B_proj = nn.Linear(rd, cd)
        self.psi_proj = nn.Linear(rd, cd)

    def forward(self, spectral_x):
        x_ri = to_real(spectral_x)  # (B, T, 272)
        A = torch.sigmoid(self.A_proj(x_ri))
        B = torch.sigmoid(self.B_proj(x_ri))
        psi = self.psi_proj(x_ri)
        return parallel_associative_scan(A, B * psi)

In [7]:
class SpectralInteraction(nn.Module):
    """Two-level active computation in spectral domain.
    1. Intra-subbundle: 17x17 complex mixing per channel (modes interact)
    2. Cross-subbundle: 8x8 real coupling per frequency (channels connect)
    3. Context modulation: SSM output gates which interactions activate."""

    def __init__(self, cfg):
        super().__init__()
        K, shd = cfg.n_subbundles, cfg.spectral_half_dim
        self.intra_real = nn.Parameter(
            torch.eye(shd).unsqueeze(0).repeat(K,1,1) + torch.randn(K,shd,shd)*0.02)
        self.intra_imag = nn.Parameter(torch.randn(K, shd, shd) * 0.02)
        self.cross = nn.Parameter(
            torch.eye(K).unsqueeze(0).repeat(shd,1,1) + torch.randn(shd,K,K)*0.02)
        self.ctx_gate = nn.Linear(cfg.context_dim, cfg.spectral_dim)
        self.gate = nn.Parameter(torch.tensor(-2.0))

    def forward(self, spectral_x, q_t, cfg):
        K, shd = cfg.n_subbundles, cfg.spectral_half_dim
        shape = spectral_x.shape[:-1]
        subs = spectral_x.reshape(*shape, K, shd)
        W = torch.complex(self.intra_real, self.intra_imag)
        mixed = torch.einsum('...ks,ksd->...kd', subs, W)
        mixed_t = mixed.transpose(-1, -2)
        coupled_t = torch.einsum('...fk,fkj->...fj', mixed_t, self.cross.to(mixed_t.dtype))
        coupled = coupled_t.transpose(-1, -2)
        ctx = torch.sigmoid(self.ctx_gate(q_t)).reshape(*shape, K, shd)
        coupled = coupled * ctx
        g = torch.sigmoid(self.gate)
        return (subs + g * (coupled - subs)).reshape(*shape, K * shd)

In [8]:
class SpectralTransport(nn.Module):
    """Context-dependent spectral kernel: exp(-D*w^2 - i*w*A).
    Signed diffusion (tanh). Operates purely in spectral domain."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.D_proj = nn.Linear(cfg.context_dim, cfg.spectral_dim)
        self.A_proj = nn.Linear(cfg.context_dim, cfg.spectral_dim)

    def forward(self, spectral_x, q_t):
        cfg = self.cfg
        D = torch.tanh(self.D_proj(q_t))
        A = self.A_proj(q_t)
        w = torch.fft.rfftfreq(cfg.subbundle_dim, d=1.0, device=spectral_x.device)
        w = w.repeat(cfg.n_subbundles)
        kernel = torch.exp(-D * w**2 - 1j * w * A)
        return spectral_x * kernel

In [9]:
class SpectralMLP(nn.Module):
    """Nonlinear transform of spectral coordinates.
    Operates on real+imag concatenation (272-dim real).
    The MLP learns to adjust spectral coordinates — move dots in Fourier space,
    change amplitudes, reshape the constellation."""

    def __init__(self, cfg):
        super().__init__()
        rd = cfg.spectral_real_dim  # 272
        hd = cfg.mlp_hidden         # 408
        self.fc1 = nn.Linear(rd, hd)
        self.fc2 = nn.Linear(hd, rd)
        self.drop = nn.Dropout(cfg.dropout)
        self.spectral_dim = cfg.spectral_dim

    def forward(self, spectral_x):
        x_ri = to_real(spectral_x)
        out_ri = self.drop(self.fc2(F.silu(self.fc1(x_ri))))
        return to_complex(out_ri, self.spectral_dim)

In [10]:
class V12_3Block(nn.Module):
    """Spectral-native block. No irfft. No spatial domain.

    spectral_x -> SSM(spectral) -> context
    spectral_x -> SpectralInteraction(ctx) -> Transport(ctx) ->
    SpectralNorm -> SpectralMLP -> spectral residual -> sparsify"""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.ssm = SpectralSSM(cfg)
        self.interact = SpectralInteraction(cfg)
        self.transport = SpectralTransport(cfg)
        self.norm = SpectralNorm(cfg.spectral_dim)
        self.mlp = SpectralMLP(cfg)
        self.res_gate = nn.Parameter(torch.tensor(0.5))
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, spectral_x):
        cfg = self.cfg

        # 1. SSM reads spectral coordinates -> context
        q_t = self.ssm(spectral_x)

        # 2. Context-gated mode interaction + cross-subbundle coupling
        interacted = self.interact(spectral_x, q_t, cfg)

        # 3. Context-dependent transport (reshape the constellation)
        transported = self.transport(interacted, q_t)

        # 4. Nonlinear spectral transform
        normed = self.norm(transported)
        mlp_out = self.mlp(normed)

        # 5. Spectral residual + enforce sparsity
        gate = torch.sigmoid(self.res_gate)
        spectral_out = spectral_x + gate * mlp_out
        return spectral_sparsify(spectral_out, cfg)

In [11]:
class V12_3Model(nn.Module):
    """Spectral-native CLM. Deep supervision at blocks 2, 4, 6."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embedding = SpectralTokenEmbedding(cfg)
        self.blocks = nn.ModuleList([V12_3Block(cfg) for _ in range(cfg.n_blocks)])
        self.final_norm = nn.LayerNorm(cfg.fiber_dim)
        self.decoder = nn.Sequential(
            nn.Linear(cfg.fiber_dim, cfg.fiber_dim), nn.SiLU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.fiber_dim, cfg.vocab_size),
        )
        weights = torch.zeros(cfg.n_blocks)
        for i in range(cfg.n_blocks):
            if (i + 1) % 2 == 0:
                weights[i] = (i + 1) / cfg.n_blocks
        weights[-1] = 1.0
        self.register_buffer('block_loss_weights', weights)

    def _decode(self, spectral_x):
        """spectral -> irfft -> spatial -> logits. The ONLY irfft."""
        spatial = spectral_to_spatial(spectral_x, self.cfg)
        return self.decoder(self.final_norm(spatial))

    def forward(self, token_ids):
        cfg = self.cfg
        spectral_x = self.embedding(token_ids)
        intermediate_logits = []
        for i, block in enumerate(self.blocks):
            spectral_x = block(spectral_x)
            if self.block_loss_weights[i] > 0:
                logits = self._decode(spectral_x)[:, :-1, :]
                intermediate_logits.append((logits, self.block_loss_weights[i]))
        sp = (spectral_x.abs() < 1e-6).float().mean().item()
        return intermediate_logits[-1][0], {
            'spectral_sparsity': sp,
            'intermediate_logits': intermediate_logits,
        }


# -- GPT-Nano ---------------------------------------------------------------

class GPTNano(nn.Module):
    def __init__(self, vocab_size=65, n_embd=128, n_head=4, n_layer=12,
                 block_size=512, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList()
        for _ in range(n_layer):
            self.blocks.append(nn.ModuleDict({
                'ln1': nn.LayerNorm(n_embd),
                'attn_qkv': nn.Linear(n_embd, 3 * n_embd),
                'attn_proj': nn.Linear(n_embd, n_embd),
                'ln2': nn.LayerNorm(n_embd),
                'mlp_fc1': nn.Linear(n_embd, 4 * n_embd),
                'mlp_fc2': nn.Linear(4 * n_embd, n_embd),
            }))
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.n_head = n_head
        self.n_embd = n_embd
        self.register_buffer('causal_mask',
            torch.tril(torch.ones(block_size, block_size)).view(1,1,block_size,block_size))

    def forward(self, idx):
        B, T = idx.shape
        x = self.drop(self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device)))
        hd = self.n_embd // self.n_head
        for blk in self.blocks:
            h = blk['ln1'](x)
            qkv = blk['attn_qkv'](h).reshape(B, T, 3, self.n_head, hd)
            q, k, v = qkv.unbind(2)
            q, k, v = q.transpose(1,2), k.transpose(1,2), v.transpose(1,2)
            att = (q @ k.transpose(-2,-1)) * (hd**-0.5)
            att = att.masked_fill(self.causal_mask[:,:,:T,:T]==0, float('-inf'))
            y = (F.softmax(att, dim=-1) @ v).transpose(1,2).reshape(B, T, self.n_embd)
            x = x + blk['attn_proj'](y)
            x = x + blk['mlp_fc2'](F.gelu(blk['mlp_fc1'](blk['ln2'](x))))
        return self.lm_head(self.ln_f(x))[:, :-1, :], {}


# -- SSM+MLP Only (dense spatial, no spectral domain) -----------------------

class SSMOnlyBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.A_proj = nn.Linear(cfg.fiber_dim, cfg.context_dim)
        self.B_proj = nn.Linear(cfg.fiber_dim, cfg.context_dim)
        self.psi_proj = nn.Linear(cfg.fiber_dim, cfg.context_dim)
        self.ctx_proj = nn.Linear(cfg.context_dim, cfg.fiber_dim)
        self.mlp_fc1 = nn.Linear(cfg.fiber_dim, int(cfg.fiber_dim * cfg.mlp_ratio))
        self.mlp_fc2 = nn.Linear(int(cfg.fiber_dim * cfg.mlp_ratio), cfg.fiber_dim)
        self.norm1 = nn.LayerNorm(cfg.fiber_dim)
        self.norm2 = nn.LayerNorm(cfg.fiber_dim)
        self.drop = nn.Dropout(cfg.dropout)
        self.gate = nn.Parameter(torch.tensor(0.5))

    def forward(self, x):
        A = torch.sigmoid(self.A_proj(x))
        B = torch.sigmoid(self.B_proj(x))
        psi = self.psi_proj(x)
        q = parallel_associative_scan(A, B * psi)
        h = self.norm1(x + self.ctx_proj(q))
        mlp = self.drop(self.mlp_fc2(F.silu(self.mlp_fc1(self.norm2(h)))))
        return x + torch.sigmoid(self.gate) * mlp


class SSMOnlyModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.emb = nn.Embedding(cfg.vocab_size, cfg.fiber_dim)
        self.pos = nn.Embedding(cfg.max_seq_len, cfg.fiber_dim)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([SSMOnlyBlock(cfg) for _ in range(cfg.n_blocks)])
        self.norm = nn.LayerNorm(cfg.fiber_dim)
        self.head = nn.Sequential(
            nn.Linear(cfg.fiber_dim, cfg.fiber_dim), nn.SiLU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.fiber_dim, cfg.vocab_size),
        )

    def forward(self, ids):
        x = self.drop(self.emb(ids) + self.pos(torch.arange(ids.size(1), device=ids.device)))
        for blk in self.blocks:
            x = blk(x)
        logits = self.head(self.norm(x))[:, :-1, :]
        return logits, {'spectral_sparsity': 0.0, 'intermediate_logits': [(logits, 1.0)]}


# -- Instantiate -------------------------------------------------------------
model = V12_3Model(config).to(device)
gpt_model = GPTNano(vocab_size=vocab_size, block_size=config.seq_len).to(device)
ssm_model = SSMOnlyModel(config).to(device)

def count_params(m):
    return sum(p.numel() for p in m.parameters())

print(f"V12.3:       {count_params(model):>10,} params")
print(f"GPT-Nano:    {count_params(gpt_model):>10,} params")
print(f"SSM+MLP:     {count_params(ssm_model):>10,} params")

n_e = sum(p.numel() for p in model.embedding.parameters())
n_ssm = sum(sum(p.numel() for p in b.ssm.parameters()) for b in model.blocks)
n_int = sum(sum(p.numel() for p in b.interact.parameters()) for b in model.blocks)
n_trn = sum(sum(p.numel() for p in b.transport.parameters()) for b in model.blocks)
n_mlp = sum(sum(p.numel() for p in b.mlp.parameters()) for b in model.blocks)
n_dec = sum(p.numel() for p in model.decoder.parameters()) + \
        sum(p.numel() for p in model.final_norm.parameters())
n_oth = count_params(model) - (n_e + n_ssm + n_int + n_trn + n_mlp + n_dec)
tot = count_params(model)
print(f"\nV12.3 Breakdown:")
print(f"  Embedding:           {n_e:>8,} ({100*n_e/tot:.1f}%)")
print(f"  SpectralSSM:         {n_ssm:>8,} ({100*n_ssm/tot:.1f}%)")
print(f"  SpectralInteraction: {n_int:>8,} ({100*n_int/tot:.1f}%)")
print(f"  SpectralTransport:   {n_trn:>8,} ({100*n_trn/tot:.1f}%)")
print(f"  SpectralMLP:         {n_mlp:>8,} ({100*n_mlp/tot:.1f}%)")
print(f"  Decoder:             {n_dec:>8,} ({100*n_dec/tot:.1f}%)")
print(f"  Other (norms,gates): {n_oth:>8,} ({100*n_oth/tot:.1f}%)")

V12.3:        2,418,813 params
GPT-Nano:     2,461,696 params
SSM+MLP:      2,210,631 params

V12.3 Breakdown:
  Embedding:             17,680 (0.7%)
  SpectralSSM:          628,992 (26.0%)
  SpectralInteraction:  139,542 (5.8%)
  SpectralTransport:    210,528 (8.7%)
  SpectralMLP:         1,335,792 (55.2%)
  Decoder:               83,009 (3.4%)
  Other (norms,gates):    3,270 (0.1%)


## Training

In [12]:
@torch.no_grad()
def estimate_loss(model, cfg, is_gpt=False):
    model.eval()
    results = {}
    for name, sd in [('train', train_data), ('val', val_data)]:
        tot_ce, tot_ok, tot_n, tot_sp = 0., 0, 0, 0.
        for _ in range(cfg.eval_steps):
            b = get_batch(sd, cfg)
            logits, info = model(b)
            tgt = b[:, 1:]
            ce = F.cross_entropy(logits.reshape(-1, cfg.vocab_size), tgt.reshape(-1))
            tot_ce += ce.item()
            tot_ok += (logits.argmax(-1) == tgt).sum().item()
            tot_n += tgt.numel()
            if not is_gpt:
                tot_sp += info.get('spectral_sparsity', 0.0)
        n = cfg.eval_steps
        results[name] = {
            'ce': tot_ce/n, 'acc': tot_ok/tot_n,
            'sparsity': tot_sp/n if not is_gpt else 0.0}
    model.train()
    return results


def train_model(model, cfg, label='V12.3', is_gpt=False, is_ssm_only=False):
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=0.05)
    mr = getattr(cfg, 'min_lr', 0) / cfg.learning_rate
    he = cfg.warmup_steps + getattr(cfg, 'lr_hold_steps', 0)

    def lr_fn(s):
        if s < cfg.warmup_steps: return s / max(1, cfg.warmup_steps)
        if s < he: return 1.0
        p = (s - he) / max(1, cfg.max_steps - he)
        return max(mr, 0.5 * (1.0 + math.cos(math.pi * p)))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_fn)
    hist = {'step':[], 'train_ce':[], 'val_ce':[], 'train_acc':[], 'val_acc':[],
            'train_bpc':[], 'val_bpc':[], 'sparsity':[], 'lr':[],
            'step_times':[], 'per_step_loss':[]}

    model.train()
    t0 = time.time()
    np_ = count_params(model)
    print(f"\nTraining {label}: {np_:,} params")
    print(f"Steps: {cfg.max_steps}, Batch: {cfg.batch_size}, Seq: {cfg.seq_len}")
    print('=' * 70)

    for step in range(cfg.max_steps + 1):
        if step % cfg.eval_interval == 0:
            r = estimate_loss(model, cfg, is_gpt=is_gpt)
            tr, vl = r['train'], r['val']
            hist['step'].append(step)
            hist['train_ce'].append(tr['ce']); hist['val_ce'].append(vl['ce'])
            hist['train_acc'].append(tr['acc']); hist['val_acc'].append(vl['acc'])
            hist['train_bpc'].append(tr['ce']/math.log(2))
            hist['val_bpc'].append(vl['ce']/math.log(2))
            hist['sparsity'].append(vl['sparsity'])
            hist['lr'].append(sched.get_last_lr()[0])
            sp = f" | Sp: {vl['sparsity']:.1%}" if not is_gpt else ''
            print(f"[{label}] Step {step:5d} | Train CE: {tr['ce']:.3f} | "
                  f"Val CE: {vl['ce']:.3f} | Val BPC: {vl['ce']/math.log(2):.3f} | "
                  f"Acc: {vl['acc']:.1%}{sp}")

        if step >= cfg.max_steps: break
        st = time.time()
        batch = get_batch(train_data, cfg)
        opt.zero_grad()
        logits, info = model(batch)
        tgt = batch[:, 1:]

        if is_gpt or is_ssm_only:
            loss = F.cross_entropy(logits.reshape(-1, cfg.vocab_size), tgt.reshape(-1))
        else:
            ce, tw = 0., 0.
            for bl, w in info['intermediate_logits']:
                ce += w * F.cross_entropy(bl.reshape(-1, cfg.vocab_size), tgt.reshape(-1))
                tw += w
            loss = ce / tw

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step()
        hist['step_times'].append(time.time() - st)
        hist['per_step_loss'].append(loss.item())
        if step % 500 == 0 and step > 0:
            print(f"    avg step time: {np.mean(hist['step_times'][-500:])*1000:.0f}ms")

    el = time.time() - t0
    ms = np.mean(hist['step_times']) * 1000
    print(f"\n{label} DONE in {el/60:.1f}min | BPC: {hist['val_bpc'][-1]:.3f} | "
          f"Acc: {hist['val_acc'][-1]:.1%} | {ms:.0f}ms/step")
    hist['avg_step_ms'] = ms; hist['n_params'] = np_
    return hist

In [13]:
v12_hist = train_model(model, config, label='V12.3')


Training V12.3: 2,418,813 params
Steps: 10000, Batch: 16, Seq: 512
[V12.3] Step     0 | Train CE: 4.169 | Val CE: 4.171 | Val BPC: 6.018 | Acc: 1.9% | Sp: 64.7%
[V12.3] Step   500 | Train CE: 1.633 | Val CE: 1.791 | Val BPC: 2.584 | Acc: 47.1% | Sp: 64.7%
    avg step time: 347ms


KeyboardInterrupt: 

## Baselines
GPT-Nano (12-layer attention) and SSM+MLP (no spectral domain). Same schedule.

In [ ]:
gpt_hist = train_model(gpt_model, config, label='GPT', is_gpt=True)

In [ ]:
ssm_hist = train_model(ssm_model, config, label='SSM+MLP', is_ssm_only=True)

## Results

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 15))
fig.suptitle('V12.3 (Spectral-Native) vs GPT-Nano vs SSM+MLP — seq_len=512',
             fontsize=16, fontweight='bold')
hs = [(v12_hist,'V12.3','b','o'), (gpt_hist,'GPT','r','s'), (ssm_hist,'SSM+MLP','g','^')]

for ax, key, title in [(axes[0,0],'val_ce','Val CE'), (axes[0,1],'val_bpc','Val BPC'),
                        (axes[0,2],'val_acc','Val Accuracy')]:
    for h,l,c,m in hs:
        y = [v*100 for v in h[key]] if 'acc' in key else h[key]
        ax.plot(h['step'], y, f'{c}-{m}', label=l, markersize=3)
    ax.set_xlabel('Step'); ax.set_title(title); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1,0]
w = 100
for h,l,c,m in hs:
    if len(h['per_step_loss']) > w:
        sm = np.convolve(h['per_step_loss'], np.ones(w)/w, mode='valid')
        ax.plot(range(len(sm)), sm, f'{c}-', label=l, alpha=0.8)
ax.set_title(f'Step Loss (smooth {w})'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1,1]
for h,l,c,m in hs: ax.plot(h['step'], h['lr'], f'{c}-', label=l)
ax.set_title('LR'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1,2]
ax.plot(v12_hist['step'], [s*100 for s in v12_hist['sparsity']], 'b-o', markersize=3)
ax.axhline(y=100*(1-config.spectral_sparsity/config.spectral_half_dim), color='r', ls=':', label='Target')
ax.set_title('V12.3 Spectral Sparsity'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[2,0]
for h,l,c,m in hs:
    if len(h['step_times']) > w:
        t = np.convolve([x*1000 for x in h['step_times']], np.ones(w)/w, mode='valid')
        ax.plot(range(len(t)), t, f'{c}-', label=l, alpha=0.8)
ax.set_title('ms/step'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[2,1]
for h,l,c,m in hs:
    ax.plot(h['step'], [t-v for t,v in zip(h['train_bpc'],h['val_bpc'])], f'{c}-{m}', label=l, ms=3)
ax.axhline(y=0, color='k', ls=':', alpha=0.3); ax.set_title('Gen. Gap'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[2,2]; ax.axis('off')
rows = [[l, f'{h["n_params"]:,}', f'{h["val_bpc"][-1]:.3f}', f'{h["val_acc"][-1]:.1%}', f'{h["avg_step_ms"]:.0f}']
        for h,l,_,_ in hs]
t = ax.table(cellText=rows, colLabels=['Model','Params','BPC','Acc','ms/step'], loc='center', cellLoc='center')
t.auto_set_font_size(False); t.set_fontsize(11); t.scale(1.2,1.8)
ax.set_title('Final Results', fontweight='bold', pad=20)
plt.tight_layout(); plt.savefig('v12_3_results.png', dpi=150, bbox_inches='tight'); plt.show()

print('\n' + '='*70 + '\nRESULTS\n' + '='*70)
v,g,s = v12_hist['val_bpc'][-1], gpt_hist['val_bpc'][-1], ssm_hist['val_bpc'][-1]
print(f"V12.3:    BPC {v:.3f} | Acc {v12_hist['val_acc'][-1]:.1%} | {v12_hist['avg_step_ms']:.0f}ms")
print(f"GPT-Nano: BPC {g:.3f} | Acc {gpt_hist['val_acc'][-1]:.1%} | {gpt_hist['avg_step_ms']:.0f}ms")
print(f"SSM+MLP:  BPC {s:.3f} | Acc {ssm_hist['val_acc'][-1]:.1%} | {ssm_hist['avg_step_ms']:.0f}ms")
print(f"\nSpectral contribution: {s - v:+.3f} BPC")
print('  -> SPECTRAL CONTRIBUTES!' if v < s - 0.02 else
      '  -> Spectral hurts.' if v > s + 0.02 else '  -> Marginal.')

## Diagnostics: Is Spectral Domain Active?

In [ ]:
@torch.no_grad()
def diagnostics(model, cfg):
    model.eval()
    batch = get_batch(val_data, cfg)
    sx = model.embedding(batch)
    print(f'Embedding sparsity: {(sx.abs()<1e-6).float().mean():.1%}')

    print('\n--- Per-block ---')
    for i, blk in enumerate(model.blocks):
        pre = sx.clone()
        q = blk.ssm(sx)
        post_int = blk.interact(sx, q, cfg)
        int_delta = (post_int - sx).abs().mean().item()
        int_gate = torch.sigmoid(blk.interact.gate).item()

        sx = blk(sx)
        sp = (sx.abs() < 1e-6).float().mean().item()
        mag = sx.abs().mean().item()
        gate = torch.sigmoid(blk.res_gate).item()

        # Mode diversity per subbundle
        subs = sx.reshape(*sx.shape[:-1], cfg.n_subbundles, cfg.spectral_half_dim)
        active = (subs.abs() > 1e-6).float().mean(dim=(0,1))
        div = active.std(dim=-1).mean().item()

        print(f'Block {i+1}: sp={sp:.1%} mag={mag:.4f} gate={gate:.3f} '
              f'int_delta={int_delta:.4f} int_gate={int_gate:.3f} mode_div={div:.3f}')

    print('\n--- Cross-subbundle coupling ---')
    for i, blk in enumerate(model.blocks):
        c = blk.interact.cross
        off = (c - torch.eye(cfg.n_subbundles, device=c.device).unsqueeze(0)).abs()
        print(f'  Block {i+1}: mean={off.mean():.4f} max={off.max():.4f}')

    print('\n--- Transport D (diffusion) ---')
    sx = model.embedding(batch)
    for i, blk in enumerate(model.blocks):
        q = blk.ssm(sx)
        D = torch.tanh(blk.transport.D_proj(q))
        print(f'  Block {i+1}: mean={D.mean():.3f} std={D.std():.3f} neg={100*(D<0).float().mean():.0f}%')
        sx = blk(sx)
    model.train()

diagnostics(model, config)

In [ ]:
@torch.no_grad()
def gen(model, prompt, cfg, n=200, temp=0.8, is_gpt=False):
    model.eval()
    ids = torch.tensor([stoi[c] for c in prompt], dtype=torch.long, device=device).unsqueeze(0)
    for _ in range(n):
        ctx = ids[:, -cfg.seq_len:]
        logits, _ = model(ctx)
        p = F.softmax(logits[:, -1, :] / temp, dim=-1)
        ids = torch.cat([ids, torch.multinomial(p, 1)], dim=1)
    return ''.join(itos[i.item()] for i in ids[0])

for p in ['ROMEO:\n', 'To be or not to ', 'The king ']:
    print(f"\nPrompt: {repr(p)}")
    print(f"  V12.3:   {gen(model, p, config)[len(p):len(p)+120]}")
    print(f"  GPT:     {gen(gpt_model, p, config, is_gpt=True)[len(p):len(p)+120]}")
    print(f"  SSM+MLP: {gen(ssm_model, p, config)[len(p):len(p)+120]}")

## Summary: V12.3 Spectral-Native

**Core change:** The model lives in spectral domain. No irfft inside blocks.
SSM and MLP operate directly on spectral coordinates (real+imag concatenation).
The spectral state is the only state that persists across blocks.

**Block flow:**
```
spectral_x (136 complex, sparse 6/17) ->
  SpectralSSM(real+imag) -> context (128 real) ->
  SpectralInteraction(intra 17x17 + cross 8x8, context-gated) ->
  SpectralTransport(exp(-D*w^2 - i*w*A)) ->
  SpectralNorm -> SpectralMLP(272 -> 408 -> 272) ->
  spectral residual + sparsify -> spectral_out
```

**Why this works:** Tokens are constellations of active modes in Fourier space.
The SSM learns which modes matter from sequential context.
The MLP learns to adjust spectral coordinates — move dots, change amplitudes.
Cross-subbundle coupling connects different parts of the spectral representation.
Transport provides context-dependent diffusion and advection.
Sparsification enforces the manifold constraint every block.

**Removed:** Hopfield memory, Langevin settling, all intra-block irfft/rfft.
These were spatial-domain concepts that distracted from spectral learning.

**Sequence length 512:** SSM recurrence decays over distance. Low-frequency spectral
modes carry long-range structure without decay. Division of labor:
SSM = short-range context, Spectral = long-range structure.